In [2]:
import polars as pl

CSV_PATH = "MTA_Subway_Hourly_Ridership__2020-2024_20250430.csv"

# 1. Define sólo los tipos que realmente necesitas
schema = {
    "transit_timestamp": pl.Utf8,         # lo parsearemos a Datetime después
    "transit_mode": pl.Categorical,
    "station_complex_id": pl.Utf8,        # leer como texto → luego cast a Int64 (TRAM2 = nulo)
    "station_complex": pl.Categorical,
    "borough": pl.Categorical,
    "payment_method": pl.Categorical,
    "fare_class_category": pl.Categorical,
    "ridership": pl.UInt32,
    "transfers": pl.UInt32,
    "latitude": pl.Float64,
    "longitude": pl.Float64,
    "Georeference": pl.Utf8,              # «POINT(lon lat)» en WKT → string
}

# 2. Lector perezoso
lf = pl.scan_csv(
    CSV_PATH,
    # evita gastar tiempo/ram infiriendo todo
    infer_schema_length=10000,
    schema_overrides=schema,
    null_values=["", "TRAM2"],               # TRAM2 era la cadena que te rompía el parseo
    ignore_errors=True,                      # si aparece algo raro, lo marca nulo y sigue
)


In [13]:
import io, zipfile, requests, polars as pl

GTFS_URL = "https://web.mta.info/developers/data/nyct/subway/google_transit.zip"
gtfs_bytes = requests.get(GTFS_URL, timeout=30).content

with zipfile.ZipFile(io.BytesIO(gtfs_bytes)) as z:
    routes = pl.read_csv(
        io.BytesIO(z.read("routes.txt")),
        ignore_errors=True       # GTFS usa \t o , según versión
    )

# Solo Subway (route_type == 1)
routes = routes.filter(pl.col("route_type") == 1).select(
    pl.col("route_short_name").alias("service"),
    pl.col("route_color").alias("hex_color")
)

In [ ]:
TRUNK_MAP = {
    "EE352E": ("IRT Broadway–7 Av", "IRT"),     # rojo 1-2-3
    "00933C": ("IRT Lexington Av", "IRT"),      # verde 4-5-6
    "0039A6": ("IND 8th Av",       "IND"),      # azul A-C-E
    "FF6319": ("IND 6th Av",       "IND"),      # naranja B-D-F-M
    "FCCC0A": ("BMT Broadway",     "BMT"),      # amarillo N-Q-R-W
    "A7A9AC": ("BMT Canarsie",     "BMT"),      # gris claro L
    "6CBE45": ("IND Crosstown",    "IND"),      # lima G
    "996633": ("BMT Nassau",       "BMT"),      # café J-Z
    "B933AD": ("IRT Flushing",     "IRT"),      # morado 7
    "00AEEF": ("IND 2nd Av",       "IND"),      # turquesa (Q hoy, T futuro)
    "808183": ("Shuttle",          "NA"),       # gris oscuro S
}


trunk_df = pl.DataFrame({
    "hex_color":  list(TRUNK_MAP.keys()),
    "trunk_line": [v[0] for v in TRUNK_MAP.values()],
    "division":   [v[1] for v in TRUNK_MAP.values()],
})

# 2) Asegura mayúsculas y haz left join
routes = routes.with_columns(
    pl.col("hex_color").str.to_uppercase().alias("hex_color")
).join(trunk_df, on="hex_color", how="left")





In [16]:
routes.head()

service,hex_color,trunk_line,division
str,str,str,str
"""1""","""EE352E""","""IRT Broadway–7 Av""","""IRT"""
"""2""","""EE352E""","""IRT Broadway–7 Av""","""IRT"""
"""3""","""EE352E""","""IRT Broadway–7 Av""","""IRT"""
"""4""","""00933C""","""IRT Lexington Av""","""IRT"""
"""5""","""00933C""","""IRT Lexington Av""","""IRT"""


In [2]:
from datetime import datetime

lf = (
    pl.scan_csv(
        CSV_PATH,
        schema_overrides=schema,
        infer_schema_length=10_000,
        null_values=["", "TRAM2"],
        ignore_errors=True,
    )
    # Casts y limpiezas que quieras empujar al lector
    .with_columns(
        pl.col("station_complex_id").cast(pl.Int64, strict=False)
    )
)


In [3]:
MTA_Hourly_Subway_Riderships = lf.collect(streaming=True) 

C:\Users\danhm\AppData\Local\Temp\ipykernel_17832\1583937794.py:1: DeprecationWarning: The argument `streaming` is deprecated and is being replaced by the `engine` argument.
  MTA_Hourly_Subway_Riderships = lf.collect(streaming=True)
C:\Users\danhm\AppData\Local\Temp\ipykernel_17832\1583937794.py:1: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  MTA_Hourly_Subway_Riderships = lf.collect(streaming=True)


In [4]:
MTA_Hourly_Subway_Riderships = MTA_Hourly_Subway_Riderships.with_columns(
    pl.col("transit_timestamp")
      .str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p", strict=False)
      .alias("transit_timestamp")
)
MTA_Hourly_Subway_Riderships.head()

transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,Georeference
datetime[μs],cat,i64,cat,cat,cat,cat,u32,u32,f64,f64,str
2022-11-28 13:00:00,"""subway""",126,"""Jefferson St (L)""","""Brooklyn""","""metrocard""","""Metrocard - Seniors & Disabili…",6,0,40.706608,-73.92291,"""POINT (-73.92291 40.706608)"""
2022-11-28 17:00:00,"""subway""",190,"""80 St (A)""","""Queens""","""metrocard""","""Metrocard - Fair Fare""",5,0,40.67937,-73.85899,"""POINT (-73.85899 40.67937)"""
2022-11-28 11:00:00,"""subway""",168,"""Spring St (C,E)""","""Manhattan""","""omny""","""OMNY - Seniors & Disability""",2,0,40.726227,-74.00374,"""POINT (-74.00374 40.726227)"""
2022-11-28 01:00:00,"""subway""",129,"""Halsey St (L)""","""Queens""","""metrocard""","""Metrocard - Unlimited 7-Day""",1,0,40.695602,-73.90408,"""POINT (-73.90408 40.695602)"""
2022-11-28 20:00:00,"""subway""",2,"""Astoria Blvd (N,W)""","""Queens""","""metrocard""","""Metrocard - Full Fare""",17,1,40.77026,-73.91785,"""POINT (-73.91785 40.77026)"""


In [6]:
MTA_Hourly_Subway_Riderships.write_parquet(
    "MTA_Hourly_Subway_Riderships.parquet",
    compression="zstd",            # "snappy", "gzip", "none"…
    statistics=True                # escribe metadatos de estadísticas columnares
)

In [10]:
import polars as pl

schema = {
    "GTFS Stop ID":         pl.Utf8,          # id numérico → luego cast a Int64 si procede
    "Station ID":           pl.Utf8,
    "Complex ID":           pl.Utf8,
    "Division":             pl.Categorical,
    "Line":                 pl.Categorical,
    "Stop Name":            pl.Categorical,
    "Borough":              pl.Categorical,
    "CBD":                  pl.Boolean,       # Y/N → conviértelo a bool tras la carga
    "Daytime Routes":       pl.Utf8,          # rutas separadas por espacio
    "Structure":            pl.Categorical,
    "GTFS Latitude":        pl.Float64,
    "GTFS Longitude":       pl.Float64,
    "North Direction Label": pl.Categorical,
    "South Direction Label": pl.Categorical,

    # ------ Accesibilidad ------
    # 0 = no accesible, 1 = totalmente accesible, 2 = parcialmente accesible
    "ADA":                  pl.UInt8,

    # Para estos dos campos lo habitual es 0/1 (“no”/“sí”).
    # Si el fichero trae Y/N o texto, cárgalos como Utf8 y castea después.
    "ADA Northbound":       pl.UInt8,         # 0 = no, 1 = accesible
    "ADA Southbound":       pl.UInt8,         # 0 = no, 1 = accesible

    "ADA Notes":            pl.Utf8,
    "Georeference":         pl.Utf8,          # «POINT(lon lat)» en WKT
}

CSV_PATH = "MTA_Subway_Stations.csv"

lf = (
    pl.scan_csv(
        CSV_PATH,
        schema_overrides=schema,
        infer_schema_length=10_000,
        null_values=["", "TRAM2"],
        ignore_errors=True,
    )
    # Casts y limpiezas que quieras empujar al lector
    .with_columns(
        pl.col("GTFS Stop ID").cast(pl.Int64, strict=False),
        pl.col("Station ID").cast(pl.Int64, strict=False),
        pl.col("Complex ID").cast(pl.Int64, strict=False)
    )
)


MTA_Subway_Stations= lf.collect(streaming=True) 

MTA_Subway_Stations.write_parquet(
    "MTA_Subway_Stations.parquet",
    compression="zstd",            
    statistics=True               
)



C:\Users\danhm\AppData\Local\Temp\ipykernel_7600\1145889518.py:51: DeprecationWarning: The argument `streaming` is deprecated and is being replaced by the `engine` argument.
  MTA_Subway_Stations= lf.collect(streaming=True)
C:\Users\danhm\AppData\Local\Temp\ipykernel_7600\1145889518.py:51: DeprecationWarning: The old streaming engine is being deprecated and will soon be replaced by the new streaming engine. Starting Polars version 1.23.0 and until the new streaming engine is released, the old streaming engine may become less usable. For people who rely on the old streaming engine, it is suggested to pin your version to before 1.23.0.

More information on the new streaming engine: https://github.com/pola-rs/polars/issues/20947
  MTA_Subway_Stations= lf.collect(streaming=True)


In [5]:
schema = {
    # ──────────── Claves temporales ────────────
    "Year":           pl.UInt16,      # 2020, 2021…  → entero corto
    "Month":          pl.UInt8,       # 1-12
    "Day of Week":    pl.Categorical, # Monday, Tuesday…
    "Hour of Day":    pl.UInt8,       # 0-23
    "Timestamp":      pl.Datetime,    # representativo del bloque año-mes-dow-hora

    # ──────────── Origen ────────────
    "Origin Station Complex ID":   pl.Utf8,        # «station_complex_id» textual (A14, 635…)
    "Origin Station Complex Name": pl.Categorical, # nombre de la estación
    "Origin Latitude":             pl.Float64,
    "Origin Longitude":            pl.Float64,

    # ──────────── Destino ────────────
    "Destination Station Complex ID":   pl.Utf8,
    "Destination Station Complex Name": pl.Categorical,
    "Destination Latitude":             pl.Float64,
    "Destination Longitude":            pl.Float64,

    # ──────────── Métrica ────────────
    "Estimated Average Ridership": pl.Float64,  # valor medio; puede no ser entero
}

CSV_PATH = "MTA_Subway_Origin-Destination_Ridership_Estimate__2024.csv"


pl.scan_csv(CSV_PATH, schema_overrides=schema, infer_schema_length=10_000, null_values=["", "TRAM2"], ignore_errors=True, low_memory=True,
            rechunk=False).with_columns(
    pl.col("Origin Station Complex ID").cast(pl.UInt32, strict=False),
    pl.col("Destination Station Complex ID").cast(pl.UInt32, strict=False)
).with_columns(
    pl.col("Timestamp")
      .str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p", strict=False)
      .alias("Timestamp")
).sink_parquet("MTA_OD_2024.parquet")

SchemaError: invalid series dtype: expected `String`, got `datetime[μs]` for series with name `Timestamp`

In [3]:
DF_MTA_Origin_Destination_Subway = pl.read_parquet("MTA_OD_2024.parquet")

In [4]:
DF_MTA_Origin_Destination_Subway.head()

Year,Month,Day of Week,Hour of Day,Timestamp,Origin Station Complex ID,Origin Station Complex Name,Origin Latitude,Origin Longitude,Destination Station Complex ID,Destination Station Complex Name,Destination Latitude,Destination Longitude,Estimated Average Ridership,Origin Point,Destination Point
u16,u8,cat,u8,datetime[μs],u32,cat,f64,f64,u32,cat,f64,f64,f64,str,str
2024,4,"""Monday""",0,null,299,"""Dyckman St (1)""",40.860531,-73.925536,271,"""Steinway St (M,R)""",40.756879,-73.92074,0.564,"""POINT (-73.925536 40.860531)""","""POINT (-73.92074 40.756879)"""
2024,4,"""Thursday""",11,null,432,"""Prospect Av (2,5)""",40.819585,-73.90177,273,"""Queens Plaza (E,M,R)""",40.748973,-73.937243,1.0728,"""POINT (-73.90177 40.819585)""","""POINT (-73.937243 40.748973)"""
2024,4,"""Monday""",0,null,604,"""161 St-Yankee Stadium (B,D,4)""",40.82795,-73.925741,601,"""14 St (F,M,1,2,3)/6 Av (L)""",40.737796,-73.997732,0.9866,"""POINT (-73.925741 40.82795)""","""POINT (-73.997732 40.737796)"""
2024,4,"""Thursday""",11,null,614,"""59 St-Columbus Circle (A,B,C,D…",40.768272,-73.981833,265,"""Grand Av-Newtown (M,R)""",40.737015,-73.877223,2.1482,"""POINT (-73.981833 40.768272)""","""POINT (-73.877223 40.737015)"""
2024,4,"""Monday""",1,null,167,"""W 4 St-Wash Sq (A,C,E,B,D,F,M)""",40.732338,-74.000495,97,"""Myrtle Av (M,J,Z)""",40.697207,-73.935657,1.4772,"""POINT (-74.000495 40.732338)""","""POINT (-73.935657 40.697207)"""


### CORRER A PARTIR DE AQUI DESPUES DE HABER CREADO TODOS LOS PARQUETS

In [ ]:
DF_MTA_Hourly_Subway_Riderships = pl.read_parquet("MTA_Hourly_Subway_Riderships.parquet")

In [3]:
DF_MTA_Hourly_Subway_Riderships.head()

transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,Georeference
datetime[μs],cat,i64,cat,cat,cat,cat,u32,u32,f64,f64,str
2022-11-28 13:00:00,"""subway""",126,"""Jefferson St (L)""","""Brooklyn""","""metrocard""","""Metrocard - Seniors & Disabili…",6,0,40.706608,-73.92291,"""POINT (-73.92291 40.706608)"""
2022-11-28 17:00:00,"""subway""",190,"""80 St (A)""","""Queens""","""metrocard""","""Metrocard - Fair Fare""",5,0,40.67937,-73.85899,"""POINT (-73.85899 40.67937)"""
2022-11-28 11:00:00,"""subway""",168,"""Spring St (C,E)""","""Manhattan""","""omny""","""OMNY - Seniors & Disability""",2,0,40.726227,-74.00374,"""POINT (-74.00374 40.726227)"""
2022-11-28 01:00:00,"""subway""",129,"""Halsey St (L)""","""Queens""","""metrocard""","""Metrocard - Unlimited 7-Day""",1,0,40.695602,-73.90408,"""POINT (-73.90408 40.695602)"""
2022-11-28 20:00:00,"""subway""",2,"""Astoria Blvd (N,W)""","""Queens""","""metrocard""","""Metrocard - Full Fare""",17,1,40.77026,-73.91785,"""POINT (-73.91785 40.77026)"""


In [11]:
DF_MTA_Subway_Stations = pl.read_parquet("MTA_Subway_Stations.parquet")
DF_MTA_Subway_Stations.head()

GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference
i64,i64,i64,cat,cat,cat,cat,bool,str,cat,f64,f64,cat,cat,u8,u8,u8,str,str
127,317,611,"""IRT""","""Broadway - 7Av""","""Times Sq-42 St""","""M""",true,"""1 2 3""","""Subway""",40.75529,-73.987495,"""Uptown""","""Downtown""",1,1,1,null,"""POINT (-73.987495 40.75529)"""
null,515,515,"""SIR""","""Staten Island""","""Annadale""","""SI""",false,"""SIR""","""Open Cut""",40.54046,-74.178217,"""Ferry""","""South Shore""",0,0,0,null,"""POINT (-74.178217 40.54046)"""
null,139,627,"""BMT""","""Franklin Shuttle""","""Franklin Av""","""Bk""",false,"""S""","""Elevated""",40.680596,-73.955827,"""Last Stop""","""Prospect Park""",1,1,1,null,"""POINT (-73.955827 40.680596)"""
254,349,349,"""IRT""","""Eastern Pky""","""Junius St""","""Bk""",false,"""3""","""Elevated""",40.663515,-73.902447,"""Manhattan""","""New Lots""",0,0,0,null,"""POINT (-73.902447 40.663515)"""
null,108,108,"""BMT""","""Myrtle Av""","""Middle Village-Metropolitan Av""","""Q""",false,"""M""","""Elevated""",40.711396,-73.889601,"""Inbound""","""Last Stop""",1,1,1,null,"""POINT (-73.889601 40.711396)"""


In [12]:
unique_MTA_stations = DF_MTA_Subway_Stations["Line"].unique()

for station in unique_MTA_stations:
    print(station)

Broadway - 7Av
Staten Island
Franklin Shuttle
Eastern Pky
Myrtle Av
Flushing
Pelham
Jamaica
6th Av - Culver
4th Av
8th Av - Fulton St
Concourse
Lexington Av
Lenox - White Plains Rd
Sea Beach
Astoria
West End
Rockaway
Queens Blvd
Broadway - Brighton
Dyre Av
Jerome Av
Broadway
Nostrand
Canarsie
Crosstown
Liberty Av
Clark St
Second Av
63rd St
Sea Beach / West End / Culver / Brighton
Queens - Archer
Lexington - Shuttle
Manhattan Bridge


In [4]:
min_date = DF_MTA_Hourly_Subway_Riderships["transit_timestamp"].min()
max_date = DF_MTA_Hourly_Subway_Riderships["transit_timestamp"].max()
print("MIN:", min_date, " | MAX:", max_date)

MIN: 2020-07-01 00:00:00  | MAX: 2024-12-31 23:00:00


In [12]:
unique_MTA_stations = MTA_Hourly_Subway_Riderships["station_complex_id"].unique()

for station in unique_MTA_stations:
    print(station)

None
1
2
3
4
5
6
8
9
10
13
14
16
17
20
22
26
28
30
31
32
33
34
35
36
37
38
39
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
64
65
66
67
68
69
70
71
72
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
93
94
95
96
97
98
99
100
101
103
107
108
109
110
111
113
114
118
119
120
122
123
124
125
126
127
129
130
131
133
134
135
136
137
138
141
143
144
145
146
147
149
150
151
152
153
154
155
156
157
158
159
160
162
164
165
167
168
169
173
175
176
177
179
180
181
182
183
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
220
221
222
223
224
225
228
231
232
234
235
236
237
238
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
268
269
270
271
272
273
276
277
278
279
280
282
283
284
286
287
288
289
290
291
292
293
294
295
296
297
298
299
300
301
303
304
305
306
307
308
309
310
311
312
313
314
316
318
319
320
321
323
324
325
326
327
32

In [5]:
num_filas = MTA_Hourly_Subway_Riderships.height       # equivalente a len(df) o df.shape[0]
print(num_filas)

110696369
